# Cookie eye-tracking data processing

This notebook constructs recording-level eye-tracking measures, joins survey data, and exports labeled SPSS datasets. It does not run the ANOVA or mediation models.

Use **Restart Kernel and Run All Cells** after configuring the local input files below. Condition codes are **1 = Single** and **2 = Multiple**. The same five AOI positions are used in both conditions; four positions are empty in the Single display.

Input summary durations are milliseconds; raw recording timestamps are microseconds. The final analysis dataset stores durations in seconds. Participant-level tables displayed during execution should be cleared from a copy before that copy is shared.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import pyreadstat
from IPython.display import display

# Open this notebook from the repository folder.
# Change these settings if your authorized inputs are stored elsewhere.
PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / "data" / "eye_tracking"
SURVEY_PATH = (
    PROJECT_DIR / "data" / "survey"
    / "FinalDataset_ToMergeWithEyeTracking20261007.sav"
)
OUTPUT_DIR = PROJECT_DIR / "outputs"

# Report all missing inputs before starting the processing pipeline.
required_inputs = [SURVEY_PATH]
for group, prefix in [
    ("Group 1_CookieSingle", "Cookiesingle"),
    ("Group 2_CookieMultiple", "Cookiemultiple"),
]:
    folder = DATA_DIR / group
    required_inputs.extend([
        folder / f"{prefix} AOI-based.tsv",
        folder / f"{prefix} Interval-based.tsv",
        folder / f"{prefix} Data export.tsv",
        folder / "TotalFix_Visit" / f"{prefix} Fixation_Whole Image.tsv",
        folder / "TotalFix_Visit" / f"{prefix} Visit.tsv",
    ])
missing_inputs = [str(path) for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(
        "Provide the required local inputs or update the configuration:\n"
        + "\n".join(missing_inputs)
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python version:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("pyreadstat:", pyreadstat.__version__)


## 1. Aggregate native AOI and interval metrics

Import each condition and construct one row per recording.

In [ ]:
from pathlib import Path
import pandas as pd

# Load the Multiple-condition AOI export using the configuration above.


aoi_path = (
    DATA_DIR
    / "Group 2_CookieMultiple"
    / "Cookiemultiple AOI-based.tsv"
)

aoi_multi = pd.read_csv(
    aoi_path,
    sep="\t", # tsv is separated by tab instead of commas
    encoding="utf-8-sig",
    dtype={"Recording": "string"}
)

print("AOI table shape:", aoi_multi.shape)

aoi_multi[
    [
        "Recording",
        "Cookie",
        "AOI",
        "Number_of_fixations",
        "Total_duration_of_fixations"
    ]
].head()

In [ ]:
# Verify that each recording has five unique AOI rows.
assert not aoi_multi.duplicated(["Recording", "Cookie"]).any(), (
    "Duplicate recording–AOI pairs found."
)

rows_per_recording = aoi_multi.groupby("Recording").size()
assert rows_per_recording.eq(5).all(), (
    "Each recording must contain five AOI rows."
)

# Sum fixation count and duration across the five AOIs.
fixation_multi = (
    aoi_multi
    .groupby("Recording", as_index=False, sort=False)
    .agg(
        AOI_fixation_count=("Number_of_fixations", "sum"),
        AOI_fixation_duration_ms=("Total_duration_of_fixations", "sum")
    )
)

# Convert milliseconds to seconds.
fixation_multi["AOI_fixation_duration_s"] = (
    fixation_multi["AOI_fixation_duration_ms"] / 1000
)

print("Recording-level table shape:", fixation_multi.shape)
fixation_multi.head()

In [ ]:
interval_path = (
    DATA_DIR
    / "Group 2_CookieMultiple"
    / "Cookiemultiple Interval-based.tsv"
)

interval_multi = pd.read_csv(
    interval_path,
    sep="\t",
    encoding="utf-8-sig",
    dtype={"Recording": "string"}
)

# Select the required native metrics and assign descriptive names.
interval_metrics_multi = interval_multi[
    ["Recording", "Duration_of_interval", "Number_of_saccades"]
].rename(
    columns={
        "Duration_of_interval": "viewing_duration_ms",
        "Number_of_saccades": "total_saccade_count"
    }
)

interval_metrics_multi["viewing_duration_s"] = (
    interval_metrics_multi["viewing_duration_ms"] / 1000
)

# Require matching recording IDs in both tables.
assert set(fixation_multi["Recording"]) == set(
    interval_metrics_multi["Recording"]
), "The two tables contain different recording IDs."

# Join one fixation-summary row to one interval-summary row.
analysis_multi = fixation_multi.merge(
    interval_metrics_multi,
    on="Recording",
    how="left",
    validate="one_to_one"
)

print("Merged table shape:", analysis_multi.shape)
analysis_multi.head()

In [ ]:
single_dir = DATA_DIR / "Group 1_CookieSingle"

aoi_single = pd.read_csv(
    single_dir / "Cookiesingle AOI-based.tsv",
    sep="\t",
    encoding="utf-8-sig",
    dtype={"Recording": "string"}
)

interval_single = pd.read_csv(
    single_dir / "Cookiesingle Interval-based.tsv",
    sep="\t",
    encoding="utf-8-sig",
    dtype={"Recording": "string"}
)

# Verify that each recording has five unique AOI rows.
assert not aoi_single.duplicated(["Recording", "Cookie"]).any(), (
    "Duplicate recording–AOI pairs found."
)

rows_per_recording = aoi_single.groupby("Recording").size()
assert rows_per_recording.eq(5).all(), (
    "Each recording must contain five AOI rows."
)

# Aggregate fixation metrics across all five positions.
fixation_single = (
    aoi_single
    .groupby("Recording", as_index=False, sort=False)
    .agg(
        AOI_fixation_count=("Number_of_fixations", "sum"),
        AOI_fixation_duration_ms=("Total_duration_of_fixations", "sum")
    )
)

fixation_single["AOI_fixation_duration_s"] = (
    fixation_single["AOI_fixation_duration_ms"] / 1000
)

# Prepare the native interval metrics.
interval_metrics_single = interval_single[
    ["Recording", "Duration_of_interval", "Number_of_saccades"]
].rename(
    columns={
        "Duration_of_interval": "viewing_duration_ms",
        "Number_of_saccades": "total_saccade_count"
    }
)

interval_metrics_single["viewing_duration_s"] = (
    interval_metrics_single["viewing_duration_ms"] / 1000
)

assert set(fixation_single["Recording"]) == set(
    interval_metrics_single["Recording"]
), "The two tables contain different recording IDs."

# Merge into one row per recording.
analysis_single = fixation_single.merge(
    interval_metrics_single,
    on="Recording",
    how="left",
    validate="one_to_one"
)

print("Single-condition table shape:", analysis_single.shape)
analysis_single.head()

In [ ]:
# Add condition identifiers.
single_labeled = analysis_single.assign(
    condition=1,
    condition_label="Single"
)

multiple_labeled = analysis_multi.assign(
    condition=2,
    condition_label="Multiple"
)

# Stack the rows from both conditions.
analysis = pd.concat( #pd.concat() stacks rows from the two conditions
    [single_labeled, multiple_labeled],
    ignore_index=True
)

# Place the identifying columns first.
id_columns = ["Recording", "condition", "condition_label"]
measurement_columns = [
    column for column in analysis.columns
    if column not in id_columns
]
analysis = analysis[id_columns + measurement_columns]

# Check that each recording occurs only once within each condition.
assert not analysis.duplicated(["condition", "Recording"]).any(), (
    "Duplicate condition-recording pairs found."
)

print("Combined table shape:", analysis.shape)
print(analysis.groupby("condition_label", sort=False).size())

analysis.head()

## 2. Reconstruct AOI-related saccade categories

Collapse consecutive sample rows sharing event type and index. Count a category only for an immediate **Fixation → Saccade → Fixation** sequence with known, consistent fixation AOI membership. Blinks, EyeNotFound, and unclassified events remain in the event sequence.

Inter-AOI means different AOI positions; within-AOI means the same individual position; AOI–background includes both directions. Background–background movements are omitted. These reconstructed eligible counts can differ from the native total saccade metric.

In [ ]:
import numpy as np

raw_single = pd.read_csv(
    DATA_DIR / "Group 1_CookieSingle" / "Cookiesingle Data export.tsv",
    sep="\t",
    encoding="utf-8-sig",
    dtype={"Recording name": "string"},
    low_memory=False
)

raw_multi = pd.read_csv(
    DATA_DIR / "Group 2_CookieMultiple" / "Cookiemultiple Data export.tsv",
    sep="\t",
    encoding="utf-8-sig",
    dtype={"Recording name": "string"},
    low_memory=False
)

# The unnumbered Single-condition AOI corresponds to position 0.
single_hit_columns = ["AOI hit [CookieSingle - cookie]"] + [
    f"AOI hit [CookieSingle - cookie {i}]" for i in range(1, 5)
]

multi_hit_columns = [
    f"AOI hit [CookieMultiple - cookie {i}]" for i in range(5)
]

SACCADE_COLUMNS = [
    "inter_aoi_saccade_count",
    "aoi_background_saccade_count",
    "within_aoi_saccade_count",
]

def fixation_position(event_rows, hit_columns):
    """Return AOI 0-4, background -1, or None for an unknown location."""
    coordinates = event_rows[
        ["Fixation point X (MCSnorm)", "Fixation point Y (MCSnorm)"]
    ].to_numpy(dtype=float)

    # Retain rows with finite fixation coordinates.
    located_rows = event_rows.loc[
        np.isfinite(coordinates).all(axis=1)
    ]

    if located_rows.empty:
        return None

    # All located samples within one fixation should agree on AOI membership.
    patterns = located_rows[hit_columns].drop_duplicates()

    if len(patterns) != 1:
        raise ValueError("AOI membership changes within a fixation event.")

    hits = patterns.iloc[0].to_numpy()

    if not np.isin(hits, [0, 1]).all() or hits.sum() > 1:
        raise ValueError("AOI membership must be binary and non-overlapping.")

    positions = np.flatnonzero(hits == 1)

    return int(positions[0]) if len(positions) else -1

def count_saccade_categories(raw, interval_table, condition, hit_columns):
    """Count three AOI-related saccade categories with valid fixation endpoints."""
    # 2 lines keep only the raw eye-tracking samples belonging to recordings listed in your interval-based export
    eligible = set(interval_table["Recording"])
    selected = raw.loc[raw["Recording name"].isin(eligible)].copy()

    if set(selected["Recording name"]) != eligible:
        raise ValueError("Some interval recordings are missing from the raw data.")

    if not selected["Sensor"].eq("Eye Tracker").all():
        raise ValueError("Expected Eye Tracker sample rows only.")

    if selected.duplicated(
        ["Recording name", "Recording timestamp"]
    ).any():
        raise ValueError("Duplicate recording timestamps require review.")

    interval_lookup = interval_table.set_index("Recording")
    output_rows = []

    for recording, samples in selected.groupby("Recording name", sort=False):
        # ensures that the eye-tracking samples for each participant are arranged in chronological order
        samples = samples.sort_values("Recording timestamp").copy()

        # Raw timestamps are microseconds; interval times are milliseconds.
        interval = interval_lookup.loc[recording]
        start_us = interval["Start_of_interval"] * 1000
        end_us = start_us + interval["Duration_of_interval"] * 1000

        # Allow 1 ms for rounded native interval boundaries.
        if (
            samples["Recording timestamp"].iloc[0] < start_us - 1000
            or samples["Recording timestamp"].iloc[-1] > end_us + 1000
        ):
            raise ValueError(
                f"Raw samples exceed the viewing interval: {recording}"
            )

        # Collapse consecutive sample rows sharing event type and index.
        event_keys = samples[
            ["Eye movement type", "Eye movement type index"]
        ]
        samples["event_run"] = (
            event_keys.ne(event_keys.shift()).any(axis=1).cumsum()
        )

        events = []

        for _, event_rows in samples.groupby("event_run", sort=False):
            event_type = event_rows["Eye movement type"].iloc[0]
            position = (
                fixation_position(event_rows, hit_columns)
                if event_type == "Fixation" else None
            )
            events.append((event_type, position))

        counts = dict.fromkeys(SACCADE_COLUMNS, 0)

        for i in range(1, len(events) - 1):
            previous_type, previous_position = events[i - 1]
            current_type, _ = events[i]
            next_type, next_position = events[i + 1]

            # Require an explicit Fixation -> Saccade -> Fixation sequence.
            if (previous_type, current_type, next_type) != (
                "Fixation", "Saccade", "Fixation"
            ):
                continue

            if previous_position is None or next_position is None:
                continue

            # Exclude saccades with both endpoints in the background.
            if previous_position == -1 and next_position == -1:
                continue

            # Count AOI/background movements in either direction.
            elif previous_position == -1 or next_position == -1:
                category = "aoi_background_saccade_count"

            # Both endpoints are in the same individual AOI.
            elif previous_position == next_position:
                category = "within_aoi_saccade_count"

            # Endpoints are in different AOIs.
            else:
                category = "inter_aoi_saccade_count"

            counts[category] += 1

        output_rows.append({
            "Recording": recording,
            "condition": condition,
            **counts
        })

    return pd.DataFrame(output_rows)

saccades_single = count_saccade_categories(
    raw_single, interval_single, 1, single_hit_columns
)

saccades_multi = count_saccade_categories(
    raw_multi, interval_multi, 2, multi_hit_columns
)

saccade_counts = pd.concat(
    [saccades_single, saccades_multi],
    ignore_index=True
)

# Replace previous counts and remove the background-background column.
analysis = analysis.drop(
    columns=SACCADE_COLUMNS + ["background_background_saccade_count"],
    errors="ignore"
).merge(
    saccade_counts,
    on=["Recording", "condition"],
    how="left",
    validate="one_to_one"
)

assert len(analysis) == len(interval_single) + len(interval_multi), (
    "The analysis row count does not match the interval exports."
)
assert analysis[SACCADE_COLUMNS].notna().all().all()

analysis[SACCADE_COLUMNS] = analysis[SACCADE_COLUMNS].astype(int)

print("Analysis table shape:", analysis.shape)

display(
    analysis.groupby("condition_label", sort=False)[SACCADE_COLUMNS].sum()
)

In [ ]:
saccade_check = analysis[
    ["Recording", "condition_label", "total_saccade_count"]
].copy()

saccade_check["retained_saccade_count"] = (
    analysis[SACCADE_COLUMNS].sum(axis=1)
)

saccade_check["other_saccade_count"] = (
    saccade_check["total_saccade_count"]
    - saccade_check["retained_saccade_count"]
)

assert saccade_check["other_saccade_count"].ge(0).all()

## 3. Define measures and inspect the saccade audit

The native-minus-retained residual also includes ineligible or differently classified events; it is not a pure background–background count.

In [ ]:
import pandas as pd
from IPython.display import display

# Definitions for the variables currently constructed.
DEFINITIONS = {
    "Recording":
        "Recording identifier used to match tables.",

    "condition":
        "Experimental condition: 1 = Single; 2 = Multiple.",

    "condition_label":
        "Experimental condition expressed as Single or Multiple.",

    "AOI_fixation_count":
        "Sum of native fixation counts across AOI positions 0-4.",

    "AOI_fixation_duration_ms":
        "Sum of native fixation durations across AOI positions 0-4 "
        "during the viewing interval, in milliseconds. "
        "Includes partial fixations.",

    "AOI_fixation_duration_s":
        "AOI_fixation_duration_ms divided by 1000, in seconds.",

    "viewing_duration_ms":
        "Duration of the stimulus-viewing interval exported by Tobii, "
        "in milliseconds.",

    "viewing_duration_s":
        "viewing_duration_ms divided by 1000, in seconds.",

    "total_saccade_count":
        "Native Number_of_saccades exported by Tobii for the "
        "viewing interval.",

    "inter_aoi_saccade_count":
        "Reconstructed count of eligible saccades between "
        "two different AOIs among positions 0-4.",

    "within_aoi_saccade_count":
        "Reconstructed count of eligible saccades with both "
        "fixation endpoints in the same individual AOI.",

    "aoi_background_saccade_count":
        "Reconstructed count of eligible saccades between any AOI "
        "and a valid fixation outside all five AOIs, "
        "counted in either direction.",

    "retained_saccade_count":
        "Sum of inter-AOI, within-AOI, and AOI-background "
        "saccade counts.",

    "other_saccade_count":
        "Native total minus retained saccade count. Includes "
        "omitted background-background saccades and differences "
        "in event eligibility or interval boundaries.",
}

AUDIT_COLUMNS = [
    "retained_saccade_count",
    "other_saccade_count",
]

# Combine the measurements and audit fields for inspection.
analysis_table = analysis.drop(
    columns=AUDIT_COLUMNS,
    errors="ignore",
).merge(
    saccade_check[
        ["Recording", "condition_label", *AUDIT_COLUMNS]
    ],
    on=["Recording", "condition_label"],
    how="left",
    validate="one_to_one",
)

if analysis_table[AUDIT_COLUMNS].isna().any().any():
    raise ValueError("Some recordings have no matching saccade audit.")

analysis_table[AUDIT_COLUMNS] = (
    analysis_table[AUDIT_COLUMNS].astype("int64")
)

# Require a definition for every displayed variable.
undefined = set(analysis_table.columns) - set(DEFINITIONS)

if undefined:
    raise ValueError(
        f"Definitions are missing for: {sorted(undefined)}"
    )

variable_definitions = pd.DataFrame({
    "Variable": analysis_table.columns,
    "Definition": [
        DEFINITIONS[column]
        for column in analysis_table.columns
    ],
})

# Attach definitions and counting rules as table metadata.
analysis_table.attrs["variable_definitions"] = DEFINITIONS.copy()

analysis_table.attrs["aoi_rule"] = (
    "All five position AOIs are used in both conditions, including "
    "the four empty position regions in the Single condition."
)

analysis_table.attrs["saccade_rule"] = (
    "Eligible reconstructed saccades require an immediate "
    "Fixation -> Saccade -> Fixation event sequence, "
    "with both fixation locations known."
)

print("Review table shape:", analysis_table.shape)
print("Rows per condition:")
print(
    analysis_table.groupby("condition_label", sort=False).size()
)

# Display every row and column without pandas truncation.
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    print("Full analysis table:")
    display(analysis_table)

    print("Variable definitions:")
    display(variable_definitions)

## 4. Import native whole-fixation and visit summaries

The current `Fixation_Whole Image` files supply **interval-based whole-fixation metrics**, which exclude partial fixations. AOI fixation metrics include partial fixations. These definitions are retained in variable labels.

Visit measures are imported per AOI. Revisits are `max(visits - 1, 0)` for each AOI, summed across positions for the total. These are individual-AOI visits, not visits to a union of all five AOIs.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

configs = [
    (1, "Group 1_CookieSingle", "Cookiesingle", interval_single),
    (2, "Group 2_CookieMultiple", "Cookiemultiple", interval_multi),
]

source_keys = ["Recording", "TOI", "Interval"]
join_keys = ["Recording", "condition"]

visit_metrics = {
    "Number_of_Visits": "visit_count",
    "Average_duration_of_Visit": "visit_average_duration_ms",
    "Total_duration_of_Visit": "visit_total_duration_ms",
}

visit_count_columns = [f"aoi{i}_visit_count" for i in range(5)]

metric_tables = []
visit_tables = []
native_definitions = {}
fixation_types = set()


def validate_numeric(values, name, allow_missing=False):
    """Validate numbers while preserving missing visit averages."""
    values = pd.to_numeric(values, errors="raise")
    observed = values.dropna()

    if not allow_missing and values.isna().any():
        raise ValueError(f"Missing values in {name}.")

    if not np.isfinite(observed).all() or observed.lt(0).any():
        raise ValueError(f"Invalid values in {name}.")

    return values


for condition, group, filename_prefix, intervals in configs:
    folder = Path(DATA_DIR) / group / "TotalFix_Visit"

    read_options = {
        "sep": "\t",
        "encoding": "utf-8-sig",
        "dtype": {"Recording": "string"},
    }

    fixations = pd.read_csv(
        folder / f"{filename_prefix} Fixation_Whole Image.tsv",
        **read_options,
    )

    visits = pd.read_csv(
        folder / f"{filename_prefix} Visit.tsv",
        **read_options,
    )

    # Require the same recordings, TOIs, and intervals as the existing analysis.
    if intervals["Recording"].duplicated().any():
        raise ValueError("Expected one viewing interval per recording.")

    expected_keys = set(
        intervals[source_keys].itertuples(index=False, name=None)
    )

    for table in (fixations, visits):
        actual_keys = set(
            table[source_keys].itertuples(index=False, name=None)
        )

        if (
            table[source_keys].isna().any().any()
            or actual_keys != expected_keys
        ):
            raise ValueError(
                f"Recording/TOI/interval mismatch in {filename_prefix}."
            )

    if fixations["Recording"].duplicated().any():
        raise ValueError(
            "Expected one fixation-summary row per recording."
        )

    # Preserve the distinction between ordinary and whole-fixation metrics.
    ordinary_columns = {
        "Number_of_fixations",
        "Total_duration_of_fixations",
    }

    whole_columns = {
        "Number_of_whole_fixations",
        "Total_duration_of_whole_fixations",
    }

    if ordinary_columns.issubset(fixations.columns):
        if (
            "AOI" not in fixations
            or fixations["AOI"].isna().any()
            or fixations["AOI"].nunique() != 1
        ):
            raise ValueError(
                "Ordinary totals require an export of one whole-image AOI."
            )

        count_source = "Number_of_fixations"
        duration_source = "Total_duration_of_fixations"
        output_prefix = "total_fixation"
        description = (
            "for the whole-image AOI, including partial fixations"
        )

    elif whole_columns.issubset(fixations.columns):
        if "AOI" in fixations:
            raise ValueError(
                "Expected the interval-based whole-fixation export."
            )

        count_source = "Number_of_whole_fixations"
        duration_source = "Total_duration_of_whole_fixations"
        output_prefix = "total_whole_fixation"
        description = (
            "for the viewing interval, excluding partial fixations"
        )

    else:
        raise ValueError(
            "Fixation count and duration columns were not found."
        )

    fixation_types.add(output_prefix)

    count_column = f"{output_prefix}_count"
    duration_column = f"{output_prefix}_duration_ms"

    fixation_metrics = fixations[
        ["Recording", count_source, duration_source]
    ].rename(
        columns={
            count_source: count_column,
            duration_source: duration_column,
        }
    ).copy()

    for column in (count_column, duration_column):
        fixation_metrics[column] = validate_numeric(
            fixation_metrics[column], column
        )

    if fixation_metrics[count_column].mod(1).ne(0).any():
        raise ValueError("Fixation counts must be integers.")

    fixation_metrics[count_column] = (
        fixation_metrics[count_column].astype("int64")
    )

    fixation_metrics[f"{output_prefix}_duration_s"] = (
        fixation_metrics[duration_column] / 1000
    )

    native_definitions.update({
        count_column:
            f"Native {count_source} {description}.",

        duration_column:
            f"Native {duration_source} {description}, in milliseconds.",

        f"{output_prefix}_duration_s":
            f"{duration_column} divided by 1000, in seconds.",
    })

    # The unnumbered Single-condition AOI corresponds to position 0.
    position_map = (
        {"cookie": 0, **{f"cookie {i}": i for i in range(1, 5)}}
        if condition == 1
        else {f"cookie {i}": i for i in range(5)}
    )

    visits = visits[
        [*source_keys, "AOI", *visit_metrics]
    ].rename(
        columns=visit_metrics
    ).copy()

    visits["position"] = visits["AOI"].map(position_map)

    if visits["position"].isna().any():
        raise ValueError(
            "Unrecognized AOI names in the visit export."
        )

    visits["position"] = visits["position"].astype("int64")

    if visits.duplicated(["Recording", "position"]).any():
        raise ValueError("Duplicate recording-position pairs.")

    if not visits.groupby("Recording")["position"].nunique().eq(5).all():
        raise ValueError(
            "Each recording must have all five AOI positions."
        )

    # Import native durations directly. Keep missing averages at zero visits.
    for metric in visit_metrics.values():
        visits[metric] = validate_numeric(
            visits[metric],
            metric,
            allow_missing=(metric == "visit_average_duration_ms"),
        )

    if visits["visit_count"].mod(1).ne(0).any():
        raise ValueError("Visit counts must be integers.")

    visits["visit_count"] = visits["visit_count"].astype("int64")

    zero_visits = visits["visit_count"].eq(0)

    if not visits.loc[
        zero_visits, "visit_total_duration_ms"
    ].eq(0).all():
        raise ValueError(
            "Zero visits must have zero total duration."
        )

    if not visits.loc[
        zero_visits, "visit_average_duration_ms"
    ].isna().all():
        raise ValueError(
            "Average duration should be missing at zero visits."
        )

    if visits.loc[
        ~zero_visits, "visit_average_duration_ms"
    ].isna().any():
        raise ValueError(
            "A visited AOI has a missing average duration."
        )

    # Create one row per recording, with separate columns for positions 0-4.
    visit_wide = visits.pivot(
        index="Recording",
        columns="position",
        values=list(visit_metrics.values()),
    )

    visit_wide.columns = [
        f"aoi{position}_{metric}"
        for metric, position in visit_wide.columns
    ]

    visit_columns = [
        f"aoi{position}_{metric}"
        for position in range(5)
        for metric in visit_metrics.values()
    ]

    visit_wide = visit_wide[visit_columns].reset_index()

    visit_wide[visit_count_columns] = (
        visit_wide[visit_count_columns].astype("int64")
    )

    # Subtract the first visit separately for each AOI.
    # An AOI with zero or one visit has zero revisits.
    revisits = (
        visit_wide[visit_count_columns] - 1
    ).clip(lower=0)

    visit_wide["center_aoi_revisit_count"] = (
        revisits["aoi0_visit_count"]
    )

    visit_wide["total_aoi_revisit_count"] = (
        revisits.sum(axis=1)
    )

    for position in range(5):
        native_definitions.update({
            f"aoi{position}_visit_count":
                f"Native Number_of_Visits for AOI position {position}.",

            f"aoi{position}_visit_average_duration_ms":
                f"Native mean visit duration for AOI position {position}, "
                "in milliseconds. Missing when there are no visits.",

            f"aoi{position}_visit_total_duration_ms":
                f"Native total visit duration for AOI position {position}, "
                "in milliseconds.",
        })

    combined = fixation_metrics.merge(
        visit_wide,
        on="Recording",
        validate="one_to_one",
    )

    combined.insert(1, "condition", condition)
    metric_tables.append(combined)

    visits.insert(1, "condition", condition)
    visit_tables.append(visits)


if len(fixation_types) != 1:
    raise ValueError(
        "Both conditions must use the same fixation metric definition."
    )

native_definitions.update({
    "center_aoi_revisit_count":
        "max(aoi0_visit_count - 1, 0).",

    "total_aoi_revisit_count":
        "Sum of max(visit_count - 1, 0), calculated separately "
        "for AOI positions 0-4.",
})

native_metrics = pd.concat(metric_tables, ignore_index=True)

# Retain a long-format table for inspecting individual AOI measurements.
native_visits = pd.concat(visit_tables, ignore_index=True)

if (
    analysis[join_keys].isna().any().any()
    or analysis.duplicated(join_keys).any()
):
    raise ValueError(
        "Analysis recording-condition keys must be unique and non-missing."
    )

if set(
    native_metrics[join_keys].itertuples(index=False, name=None)
) != set(
    analysis[join_keys].itertuples(index=False, name=None)
):
    raise ValueError(
        "Native metrics do not match the analysis recording-condition keys."
    )

native_metrics["Recording"] = native_metrics["Recording"].astype(
    analysis["Recording"].dtype
)

# Replace earlier versions of these fields so this cell can be rerun.
fixation_output_columns = [
    f"{prefix}_{suffix}"
    for prefix in ("total_fixation", "total_whole_fixation")
    for suffix in ("count", "duration_ms", "duration_s")
]

analysis = analysis.drop(
    columns=list(native_definitions) + fixation_output_columns,
    errors="ignore",
).merge(
    native_metrics,
    on=join_keys,
    how="left",
    validate="one_to_one",
    sort=False,
)

# Update the existing variable dictionary.
DEFINITIONS.update(native_definitions)

# Refresh the review table, including the existing saccade audit.
review_metadata = analysis_table.attrs.copy()

analysis_table = analysis.drop(
    columns=AUDIT_COLUMNS,
    errors="ignore",
).merge(
    saccade_check[
        ["Recording", "condition_label", *AUDIT_COLUMNS]
    ],
    on=["Recording", "condition_label"],
    how="left",
    validate="one_to_one",
)

if analysis_table[AUDIT_COLUMNS].isna().any().any():
    raise ValueError(
        "Some recordings have no matching saccade audit."
    )

analysis_table[AUDIT_COLUMNS] = (
    analysis_table[AUDIT_COLUMNS].astype("int64")
)

variable_definitions = pd.DataFrame({
    "Variable": analysis_table.columns,
    "Definition": [
        DEFINITIONS[column]
        for column in analysis_table.columns
    ],
})

analysis_table.attrs = review_metadata

analysis_table.attrs["variable_definitions"] = {
    column: DEFINITIONS[column]
    for column in analysis_table.columns
}

analysis_table.attrs["visit_rule"] = (
    "Native visit metrics; revisits subtract the first visit "
    "separately for each AOI."
)

print("Analysis table shape:", analysis.shape)
print("Review table shape:", analysis_table.shape)

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(
        analysis_table[
            ["Recording", "condition_label", *native_definitions]
        ].head(10)
    )

    display(
        variable_definitions.loc[
            variable_definitions["Variable"].isin(native_definitions)
        ]
    )

In [ ]:
# Run after the fixation and visit metrics cell.
old_name = "center_aoi_revisit_count"
new_name = "aoi0_revisit_count"
join_keys = ["Recording", "condition"]

# Rename the imported metric.
if old_name in native_metrics.columns:
    native_metrics = native_metrics.drop(
        columns=new_name, errors="ignore"
    ).rename(columns={old_name: new_name})

# Update the variable definitions.
for definitions in (native_definitions, DEFINITIONS):
    definitions.pop(old_name, None)
    definitions[new_name] = "max(aoi0_visit_count - 1, 0)."

metric_columns = [
    column for column in native_metrics.columns
    if column not in join_keys
]


def join_native_metrics(previous_table):
    """Append native metrics by recording and condition."""
    if previous_table[join_keys].isna().any().any():
        raise ValueError("Missing recording-condition keys.")

    if set(
        previous_table[join_keys].itertuples(index=False, name=None)
    ) != set(
        native_metrics[join_keys].itertuples(index=False, name=None)
    ):
        raise ValueError("Recording-condition keys do not match.")

    metadata = previous_table.attrs.copy()

    # Replace existing metric columns so rerunning is safe.
    updated = previous_table.drop(
        columns=[old_name, *metric_columns],
        errors="ignore",
    ).merge(
        native_metrics,
        on=join_keys,
        how="left",
        validate="one_to_one",
        sort=False,
    )

    updated.attrs = metadata
    return updated


# Join the metrics to both existing tables.
analysis = join_native_metrics(analysis)
analysis_table = join_native_metrics(analysis_table)

# Refresh the definitions for the complete table.
variable_definitions = pd.DataFrame({
    "Variable": analysis_table.columns,
    "Definition": [
        DEFINITIONS[column] for column in analysis_table.columns
    ],
})

analysis_table.attrs["variable_definitions"] = dict(
    zip(
        variable_definitions["Variable"],
        variable_definitions["Definition"],
    )
)

print("Updated table shape:", analysis_table.shape)

with pd.option_context("display.max_columns", None):
    display(analysis_table.head(10))

## 5. Merge survey responses and preserve SPSS definitions

Match Recording to ExperimentID after normalizing whitespace and dash characters in temporary join keys. Original identifier values remain in the dataset. This pipeline applies no comprehension-check exclusion.

In [ ]:
# Run after the AOI 0 revisit rename cell.
from pathlib import Path
import pandas as pd
import pyreadstat
from IPython.display import display

survey_path = SURVEY_PATH

# Read original values and SPSS metadata.
survey_data, survey_metadata = pyreadstat.read_sav(
    str(survey_path),
    apply_value_formats=False,
    user_missing=True,
)


def normalize_ids(values):
    """Standardize whitespace and dashes only in the matching key."""
    normalized = values.astype("string").str.strip().str.replace(
        r"[\u2010-\u2015\u2212\uff0d]", "-", regex=True
    )

    if normalized.isna().any() or normalized.eq("").any():
        raise ValueError("Missing participant IDs.")

    if normalized.duplicated().any():
        raise ValueError(
            "Participant IDs are not unique after normalization."
        )

    return normalized


# Remove an earlier survey import so this cell can be rerun.
eye_metadata = analysis_table.attrs.copy()
previous_survey_columns = eye_metadata.get("survey_columns", [])

eye_table = analysis_table.drop(
    columns=previous_survey_columns,
    errors="ignore",
).copy()

# Prevent survey columns from overwriting eye-tracking variables.
overlap = set(eye_table.columns) & set(survey_data.columns)
if overlap:
    raise ValueError(
        f"Shared column names require renaming: {sorted(overlap)}"
    )

eye_key = normalize_ids(eye_table["Recording"])
survey_key = normalize_ids(survey_data["ExperimentID"])

# Preserve the original IDs and the eye-tracking row order.
merged_data = eye_table.assign(_merge_id=eye_key).merge(
    survey_data.assign(_merge_id=survey_key),
    on="_merge_id",
    how="left",
    validate="one_to_one",
    sort=False,
    indicator="_survey_match",
)

unmatched = merged_data.loc[
    merged_data["_survey_match"].ne("both"), "Recording"
].tolist()

if unmatched:
    raise ValueError(
        f"Eye-tracking recordings without survey data: {unmatched}"
    )

merged_data = merged_data.drop(
    columns=["_merge_id", "_survey_match"]
)

# Combine eye-tracking definitions and original SPSS variable labels.
eye_definitions = {
    column: DEFINITIONS[column]
    for column in eye_table.columns
}

survey_labels = dict(survey_metadata.column_names_to_labels)

survey_definitions = {
    column: survey_labels.get(column)
    or f"No SPSS variable label supplied for {column}."
    for column in survey_data.columns
}

combined_definitions = {
    **eye_definitions,
    **survey_definitions,
}

variable_value_labels = dict(
    eye_metadata.get("variable_value_labels", {})
)
variable_value_labels.update(
    survey_metadata.variable_value_labels
)

# Preserve definitions and SPSS metadata with the merged table.
merged_data.attrs = {
    **eye_metadata,
    "variable_definitions": combined_definitions,
    "variable_value_labels": variable_value_labels,
    "survey_variable_labels": survey_labels,
    "survey_missing_ranges": survey_metadata.missing_ranges,
    "survey_variable_measures": survey_metadata.variable_measure,
    "survey_variable_formats": survey_metadata.original_variable_types,
    "survey_columns": survey_data.columns.tolist(),
    "survey_source_file": str(survey_path),
    "survey_merge_rule": (
        "ExperimentID = Recording after whitespace/dash normalization."
    ),
}

variable_definitions = pd.DataFrame({
    "Variable": merged_data.columns,
    "Definition": [
        combined_definitions[column]
        for column in merged_data.columns
    ],
})

# Update the current tables for subsequent analysis.
DEFINITIONS.update(survey_definitions)
analysis_table = merged_data

analysis = merged_data.drop(
    columns=AUDIT_COLUMNS,
    errors="ignore",
).copy()

analysis.attrs["variable_definitions"] = {
    column: combined_definitions[column]
    for column in analysis.columns
}

print("Merged table shape:", merged_data.shape)
print("Matched eye-tracking recordings:", len(merged_data))
print(
    "Survey rows without eye-tracking data:",
    len(set(survey_key) - set(eye_key)),
)

with pd.option_context("display.max_columns", None):
    display(merged_data.head(10))

display(variable_definitions)

In [ ]:
# Export the complete merged dataset to SPSS.
from pathlib import Path
import pandas as pd
import pyreadstat

output_path = OUTPUT_DIR / "Cookie_eye_tracking_with_survey.sav"

metadata = merged_data.attrs

column_labels = {
    column: metadata["variable_definitions"][column]
    for column in merged_data.columns
}

value_labels = dict(metadata.get("variable_value_labels", {}))
value_labels["condition"] = {1: "Single", 2: "Multiple"}

# Preserve survey measurement levels.
measurement_levels = {
    column: "scale" for column in merged_data.columns
}
measurement_levels.update(
    metadata.get("survey_variable_measures", {})
)

for column in ("Recording", "condition", "condition_label", "ExperimentID"):
    measurement_levels[column] = "nominal"

# Preserve survey formats and set suitable eye-tracking display formats.
variable_formats = dict(
    metadata.get("survey_variable_formats", {})
)

for column in merged_data.columns:
    if column.endswith("_count") or column == "condition":
        variable_formats[column] = "F12.0"
    elif column.endswith(("_ms", "_s")):
        variable_formats[column] = "F16.3"

# Write all rows and columns, including labels and missing-value rules.
pyreadstat.write_sav(
    merged_data,
    str(output_path),
    file_label="Cookie eye-tracking and survey data",
    column_labels=column_labels,
    variable_value_labels=value_labels,
    missing_ranges=metadata.get("survey_missing_ranges", {}),
    variable_measure=measurement_levels,
    variable_format=variable_formats,
)

# Read the file back to verify the exported values and labels.
saved_data, saved_metadata = pyreadstat.read_sav(
    str(output_path),
    apply_value_formats=False,
    user_missing=True,
)

pd.testing.assert_frame_equal(
    merged_data.reset_index(drop=True),
    saved_data,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

assert saved_metadata.column_names_to_labels == column_labels
assert saved_metadata.variable_value_labels == value_labels
assert saved_metadata.missing_ranges == metadata.get(
    "survey_missing_ranges", {}
)

print(f"Saved: {output_path}")
print(f"Rows: {saved_data.shape[0]}; columns: {saved_data.shape[1]}")
print("Data values, variable labels, value labels, and missing-value rules verified.")

## 6. Convert to seconds and derive rates, coverage, and ratios

Count rates use stimulus viewing duration in seconds. Ratios are missing when their denominator is zero. AOI fixation metrics divided by whole-fixation totals can exceed 1 because their timing definitions differ; interpret these as ratios to whole-fixation totals, not bounded proportions.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat

output_path = OUTPUT_DIR / "Cookie_eye_tracking_with_survey_seconds.sav"

data = merged_data.copy()
metadata = merged_data.attrs.copy()
definitions = dict(metadata["variable_definitions"])
survey_columns = set(metadata.get("survey_columns", []))

# Convert all millisecond fields to seconds.
ms_columns = [
    column for column in data.columns
    if column.endswith("_ms")
]
seconds_names = {
    column: column[:-3] + "_s"
    for column in ms_columns
}

for ms_column, seconds_column in seconds_names.items():
    data[seconds_column] = (
        pd.to_numeric(data[ms_column], errors="raise") / 1000
    )
    definitions[seconds_column] = (
        definitions[ms_column].replace("milliseconds", "seconds")
        + " Converted from milliseconds by dividing by 1000."
    )

data = data.drop(columns=ms_columns)

# Validate the denominator used for event rates.
viewing_seconds = pd.to_numeric(
    data["viewing_duration_s"], errors="raise"
)

if not (
    np.isfinite(viewing_seconds) & viewing_seconds.gt(0)
).all():
    raise ValueError(
        "Viewing duration must be nonmissing, finite, and positive."
    )

# Calculate per-second rates for eye-tracking counts.
count_columns = [
    column for column in data.columns
    if column.endswith("_count") and column not in survey_columns
]
rate_columns = {
    column: column + "_per_s"
    for column in count_columns
}

for count_column, rate_column in rate_columns.items():
    data[rate_column] = data[count_column] / viewing_seconds
    definitions[rate_column] = (
        f"{count_column} divided by stimulus viewing duration "
        "(viewing_duration_s), in counts per second."
    )

# Place each rate immediately after its count.
# Deduplicate seconds fields that already existed.
column_order = []

for column in merged_data.columns:
    column = seconds_names.get(column, column)

    if column in rate_columns.values() or column in column_order:
        continue

    column_order.append(column)

    if column in rate_columns:
        column_order.append(rate_columns[column])

data = data[column_order]
definitions = {
    column: definitions[column]
    for column in data.columns
}

data.attrs = {
    **metadata,
    "variable_definitions": definitions,
}
data.attrs["rate_denominator"] = (
    "Stimulus viewing duration in seconds."
)

# Update the current tables and variable definitions.
merged_data = data
analysis_table = merged_data
DEFINITIONS.update(definitions)

variable_definitions = pd.DataFrame({
    "Variable": merged_data.columns,
    "Definition": [
        definitions[column] for column in merged_data.columns
    ],
})

audit_fields = [
    *AUDIT_COLUMNS,
    *[column + "_per_s" for column in AUDIT_COLUMNS],
]
analysis = merged_data.drop(
    columns=audit_fields, errors="ignore"
).copy()
analysis.attrs["variable_definitions"] = {
    column: definitions[column]
    for column in analysis.columns
}

# Preserve SPSS category labels and survey measurement levels.
value_labels = dict(metadata.get("variable_value_labels", {}))
value_labels["condition"] = {1: "Single", 2: "Multiple"}

measurement_levels = {
    column: "scale" for column in merged_data.columns
}
measurement_levels.update(
    metadata.get("survey_variable_measures", {})
)

for column in ("Recording", "condition", "condition_label", "ExperimentID"):
    measurement_levels[column] = "nominal"

# Set display formats without rounding the stored values.
variable_formats = dict(
    metadata.get("survey_variable_formats", {})
)

for column in merged_data.columns:
    if column in rate_columns.values():
        variable_formats[column] = "F16.6"
    elif column.endswith("_count") or column == "condition":
        variable_formats[column] = "F12.0"
    elif column.endswith("_s"):
        variable_formats[column] = "F16.3"

# Export the complete dataset.
pyreadstat.write_sav(
    merged_data,
    str(output_path),
    file_label="Cookie eye-tracking and survey data: seconds and rates",
    column_labels=definitions,
    variable_value_labels=value_labels,
    missing_ranges=metadata.get("survey_missing_ranges", {}),
    variable_measure=measurement_levels,
    variable_format=variable_formats,
)

# Verify the saved values, column order, and labels.
saved_data, saved_metadata = pyreadstat.read_sav(
    str(output_path),
    apply_value_formats=False,
    user_missing=True,
)

pd.testing.assert_frame_equal(
    merged_data.reset_index(drop=True),
    saved_data,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

assert saved_metadata.column_names_to_labels == definitions
assert saved_metadata.variable_value_labels == value_labels
assert saved_metadata.missing_ranges == metadata.get(
    "survey_missing_ranges", {}
)

print(f"Saved: {output_path}")
print(f"Rows: {saved_data.shape[0]}; columns: {saved_data.shape[1]}")
print("Millisecond columns removed; rates placed immediately after counts.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat

output_path = OUTPUT_DIR / "Cookie_eye_tracking_with_survey_final.sav"

key_columns = ["Recording", "condition"]
flag_columns = [f"aoi{i}_fixated" for i in range(5)]
coverage_column = "n_aois_fixated"

ratio_specs = {
    "inter_aoi_saccade_ratio": (
        "inter_aoi_saccade_count",
        "total_saccade_count",
    ),
    "aoi_fixation_count_ratio": (
        "AOI_fixation_count",
        "total_whole_fixation_count",
    ),
    "aoi_fixation_duration_ratio": (
        "AOI_fixation_duration_s",
        "total_whole_fixation_duration_s",
    ),
}

new_columns = [
    *flag_columns,
    coverage_column,
    *ratio_specs,
]

# Obtain native fixation counts for the five AOI positions.
aoi_fixations = pd.concat([
    aoi_single[
        ["Recording", "Cookie", "Number_of_fixations"]
    ].assign(condition=1),
    aoi_multi[
        ["Recording", "Cookie", "Number_of_fixations"]
    ].assign(condition=2),
], ignore_index=True)

aoi_fixations["Cookie"] = pd.to_numeric(
    aoi_fixations["Cookie"], errors="raise"
)

if not aoi_fixations["Cookie"].isin(range(5)).all():
    raise ValueError("AOI positions must be integers from 0 to 4.")

aoi_fixations["Cookie"] = aoi_fixations["Cookie"].astype("int64")

counts = pd.to_numeric(
    aoi_fixations["Number_of_fixations"], errors="raise"
)

if (
    counts.isna().any()
    or not np.isfinite(counts).all()
    or counts.lt(0).any()
    or counts.mod(1).ne(0).any()
):
    raise ValueError(
        "Native fixation counts must be nonmissing nonnegative integers."
    )

aoi_fixations["Number_of_fixations"] = counts

if aoi_fixations.duplicated([*key_columns, "Cookie"]).any():
    raise ValueError("Duplicate recording-condition-AOI rows.")

counts_wide = aoi_fixations.pivot(
    index=key_columns,
    columns="Cookie",
    values="Number_of_fixations",
).reindex(columns=range(5))

if counts_wide.isna().any().any():
    raise ValueError(
        "Each participant must have fixation counts for all five AOIs."
    )

# 0 = no recorded fixation; 1 = at least one recorded fixation.
flags = counts_wide.gt(0).astype("int64")
flags.columns = flag_columns
flags.columns.name = None
flags[coverage_column] = flags[flag_columns].sum(axis=1)
flags = flags.reset_index()

flags["Recording"] = flags["Recording"].astype(
    merged_data["Recording"].dtype
)

if set(
    flags[key_columns].itertuples(index=False, name=None)
) != set(
    merged_data[key_columns].itertuples(index=False, name=None)
):
    raise ValueError(
        "AOI fixation keys do not match the merged dataset."
    )

metadata = merged_data.attrs.copy()
definitions = dict(metadata["variable_definitions"])

# Replace existing versions so this cell can be rerun.
data = merged_data.drop(
    columns=new_columns, errors="ignore"
).merge(
    flags,
    on=key_columns,
    how="left",
    validate="one_to_one",
    sort=False,
)

# Calculate ratios; zero denominators produce missing values.
for ratio_column, (
    numerator_column, denominator_column
) in ratio_specs.items():

    numerator = pd.to_numeric(
        data[numerator_column], errors="raise"
    )
    denominator = pd.to_numeric(
        data[denominator_column], errors="raise"
    )

    data[ratio_column] = numerator / denominator.where(
        denominator.gt(0)
    )

    definitions[ratio_column] = (
        f"{numerator_column} divided by {denominator_column}. "
        "Missing when the denominator is zero."
    )
# Sum visits across the five individual AOIs.
visit_count_columns = [
    f"aoi{i}_visit_count" for i in range(5)
]
total_visit_columns = [
    "total_aoi_visit_count",
    "total_aoi_visit_count_per_s",
]

visit_counts = data[visit_count_columns].apply(
    pd.to_numeric, errors="raise"
)

if (
    visit_counts.isna().any().any()
    or not np.isfinite(visit_counts).all().all()
    or visit_counts.lt(0).any().any()
    or visit_counts.mod(1).ne(0).any().any()
):
    raise ValueError(
        "AOI visit counts must be nonmissing nonnegative integers."
    )

# Replace previous totals so rerunning does not duplicate columns.
data = data.drop(columns=total_visit_columns, errors="ignore")

data["total_aoi_visit_count"] = (
    visit_counts.sum(axis=1).astype("int64")
)
data["total_aoi_visit_count_per_s"] = (
    data["total_aoi_visit_count"]
    / data["viewing_duration_s"].where(
        data["viewing_duration_s"].gt(0)
    )
)

new_columns.extend(total_visit_columns)

definitions["total_aoi_visit_count"] = (
    "Sum of native visit counts across individual AOI positions 0-4; "
    "includes each AOI's first visit and subsequent revisits."
)
definitions["total_aoi_visit_count_per_s"] = (
    "total_aoi_visit_count divided by viewing_duration_s, "
    "in AOI visits per stimulus-viewing second."
)

# Define and label the new AOI variables.
for position, column in enumerate(flag_columns):
    definitions[column] = (
        f"AOI position {position}: 0 = no native fixation; "
        "1 = at least one native fixation "
        "(Number_of_fixations > 0)."
    )

definitions[coverage_column] = (
    "Number of AOI positions 0-4 with at least one native fixation; "
    "sum of aoi0_fixated through aoi4_fixated, ranging from 0 to 5."
)

value_labels = dict(
    metadata.get("variable_value_labels", {})
)
value_labels["condition"] = {1: "Single", 2: "Multiple"}

for column in flag_columns:
    value_labels[column] = {0: "No", 1: "Yes"}

definitions = {
    column: definitions[column]
    for column in data.columns
}

data.attrs = {
    **metadata,
    "variable_definitions": definitions,
    "variable_value_labels": value_labels,
}

# Update the current tables and definitions.
merged_data = data
analysis_table = merged_data
DEFINITIONS.update(definitions)

variable_definitions = pd.DataFrame({
    "Variable": merged_data.columns,
    "Definition": [
        definitions[column] for column in merged_data.columns
    ],
})

audit_fields = [
    *AUDIT_COLUMNS,
    *[column + "_per_s" for column in AUDIT_COLUMNS],
]

analysis = merged_data.drop(
    columns=audit_fields, errors="ignore"
).copy()

analysis.attrs["variable_definitions"] = {
    column: definitions[column]
    for column in analysis.columns
}

# Preserve survey settings and specify measurement levels.
measurement_levels = {
    column: "scale" for column in merged_data.columns
}
measurement_levels.update(
    metadata.get("survey_variable_measures", {})
)

for column in (
    "Recording",
    "condition",
    "condition_label",
    "ExperimentID",
    *flag_columns,
):
    measurement_levels[column] = "nominal"

variable_formats = dict(
    metadata.get("survey_variable_formats", {})
)

for column in merged_data.columns:
    if column in ratio_specs or column.endswith("_per_s"):
        variable_formats[column] = "F16.6"
    elif (
        column in flag_columns
        or column == coverage_column
        or column.endswith("_count")
        or column == "condition"
    ):
        variable_formats[column] = "F12.0"
    elif column.endswith("_s"):
        variable_formats[column] = "F16.3"

# Export the complete updated dataset.
pyreadstat.write_sav(
    merged_data,
    str(output_path),
    file_label="Cookie eye-tracking and survey data: final analysis dataset",
    column_labels=definitions,
    variable_value_labels=value_labels,
    missing_ranges=metadata.get("survey_missing_ranges", {}),
    variable_measure=measurement_levels,
    variable_format=variable_formats,
)

# Verify the exported values and metadata.
saved_data, saved_metadata = pyreadstat.read_sav(
    str(output_path),
    apply_value_formats=False,
    user_missing=True,
)

pd.testing.assert_frame_equal(
    merged_data.reset_index(drop=True),
    saved_data,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

assert saved_metadata.column_names_to_labels == definitions
assert saved_metadata.variable_value_labels == value_labels
assert saved_metadata.missing_ranges == metadata.get(
    "survey_missing_ranges", {}
)

print(f"Saved: {output_path}")
print(f"Rows: {saved_data.shape[0]}; columns: {saved_data.shape[1]}")

display(
    merged_data[
        ["Recording", "condition_label", *new_columns]
    ].head(10)
)

## 7. Export short variable names and combined visit durations

SPSS variable names use at most eight characters. The combined mean visit duration is the sum of the five AOI visit durations divided by their total visit count. It is missing when no visits are recorded. The final output is `outputs/Cookie_eye_tracking_with_survey_visits.sav`.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import pyreadstat
from IPython.display import display

study_dir = OUTPUT_DIR
source_path = study_dir / "Cookie_eye_tracking_with_survey_final.sav"
output_path = study_dir / "Cookie_eye_tracking_with_survey_8char.sav"
name_key_path = study_dir / "Cookie_eye_tracking_variable_names.csv"

# Read numeric codes and retain user-defined missing-value rules.
source_data, source_meta = pyreadstat.read_sav(
    str(source_path),
    apply_value_formats=False,
    user_missing=True,
)

# N = count; PS = count per second; S = seconds; Rt = ratio.
short_names = {
    "Recording": "RecID",
    "condition": "Cond",
    "condition_label": "CondLbl",
    "AOI_fixation_count": "AOIFixN",
    "AOI_fixation_count_per_s": "AOIFixPS",
    "AOI_fixation_duration_s": "AOIFixS",
    "viewing_duration_s": "ViewS",

    "total_saccade_count": "TotSacN",
    "total_saccade_count_per_s": "TotSacPS",
    "inter_aoi_saccade_count": "IntSacN",
    "inter_aoi_saccade_count_per_s": "IntSacPS",
    "aoi_background_saccade_count": "AoiBgSN",
    "aoi_background_saccade_count_per_s": "AoiBgSPS",
    "within_aoi_saccade_count": "WthSacN",
    "within_aoi_saccade_count_per_s": "WthSacPS",
    "retained_saccade_count": "RetSacN",
    "retained_saccade_count_per_s": "RetSacPS",
    "other_saccade_count": "OthSacN",
    "other_saccade_count_per_s": "OthSacPS",

    # Preserve the native whole-fixation definitions in the variable labels.
    "total_whole_fixation_count": "TotFixN",
    "total_whole_fixation_count_per_s": "TotFixPS",
    "total_whole_fixation_duration_s": "TotFixS",

    "aoi0_revisit_count": "A0RevN",
    "aoi0_revisit_count_per_s": "A0RevPS",
    "total_aoi_revisit_count": "AOIRevN",
    "total_aoi_revisit_count_per_s": "AOIRevPS",
    "total_aoi_visit_count": "AOIVisN",
    "total_aoi_visit_count_per_s": "AOIVisPS",

    "n_aois_fixated": "NAOIFix",
    "inter_aoi_saccade_ratio": "IntSacRt",
    "aoi_fixation_count_ratio": "AOIFxNRt",
    "aoi_fixation_duration_ratio": "AOIFxDRt",

    # Original survey names must match the source file exactly.
    "作答ID": "RespID",
    "用户ID": "UserID",
    "开始时间": "StartTm",
    "结束时间": "EndTm",
    "作答总时长": "SurvDurS",
    "作答渠道": "Channel",
    "发布ID": "PublID",
    "问卷发布名称": "SurvName",
    "经度": "Lon",
    "纬度": "Lat",
    "省份": "Province",
    "城市": "City",
    "设备类型": "Device",
    "操作系统类型": "OS",
    "浏览器类型": "Browser",
    "屏幕分辨率": "ScrRes",

    "Q": "AdImage",
    "ManiC": "ManCheck",
    "Comprehen": "CmpCheck",
    "Freq": "EatFreq",
    "StudentID": "StudID",
    "ExperimentID": "ExpID",
    "DisplayQ": "DispQ",
    "DisplaQ2": "DispQ2",
    "NonRepeatRespondent": "NonRep",
    "SecondReponse": "Resp2",
    "FullDataset": "FullData",
    "Qualifier": "Qualify",
    "ExcludedDataset": "UseData",
    "filter_$": "Filter",
}

# Apply consistent names to AOI positions 0-4.
for i in range(5):
    short_names.update({
        f"aoi{i}_visit_count": f"A{i}VisN",
        f"aoi{i}_visit_count_per_s": f"A{i}VisPS",
        f"aoi{i}_visit_average_duration_s": f"A{i}VisAvS",
        f"aoi{i}_visit_total_duration_s": f"A{i}VisS",
        f"aoi{i}_fixated": f"A{i}FixYN",
    })

# Retain existing recognizable short names, such as Cal, Img1 and Age.
rename_map = {
    name: short_names.get(name, name)
    for name in source_data.columns
}

# Require valid names of at most eight characters.
invalid_names = {
    old: new
    for old, new in rename_map.items()
    if re.fullmatch(r"[A-Za-z][A-Za-z0-9_]{0,7}", new) is None
}
if invalid_names:
    raise ValueError(
        f"Add valid aliases of 1-8 characters: {invalid_names}"
    )

# SPSS variable names must be unique, ignoring capitalization.
if len({name.casefold() for name in rename_map.values()}) != len(rename_map):
    raise ValueError(
        "Short variable names must be unique, ignoring case."
    )

spss_data = source_data.rename(columns=rename_map).copy()


# Rename metadata keys while retaining their contents.
def rename_metadata(mapping):
    return {
        rename_map[name]: value
        for name, value in (mapping or {}).items()
        if name in rename_map
    }


labels = rename_metadata(source_meta.column_names_to_labels)
value_labels = rename_metadata(source_meta.variable_value_labels)
missing_rules = rename_metadata(source_meta.missing_ranges)
measures = rename_metadata(source_meta.variable_measure)
formats = rename_metadata(source_meta.original_variable_types)
display_widths = rename_metadata(source_meta.variable_display_width)

spss_data.attrs = {
    "variable_definitions": labels,
    "variable_value_labels": value_labels,
    "missing_ranges": missing_rules,
    "variable_measures": measures,
    "variable_formats": formats,
}

# Export the dataset with its full original definitions and SPSS settings.
pyreadstat.write_sav(
    spss_data,
    str(output_path),
    file_label=source_meta.file_label or "Cookie eye-tracking and survey data",
    column_labels=labels,
    variable_value_labels=value_labels,
    missing_ranges=missing_rules,
    variable_measure=measures,
    variable_format=formats,
    variable_display_width=display_widths,
    note=source_meta.notes or None,
)

# Read the export back and verify every value and the metadata.
saved_data, saved_meta = pyreadstat.read_sav(
    str(output_path),
    apply_value_formats=False,
    user_missing=True,
)

pd.testing.assert_frame_equal(
    spss_data.reset_index(drop=True),
    saved_data,
    check_dtype=False,
    check_exact=True,
)

for attribute, expected in {
    "column_names_to_labels": labels,
    "variable_value_labels": value_labels,
    "missing_ranges": missing_rules,
    "variable_measure": measures,
    "original_variable_types": formats,
    "variable_display_width": display_widths,
}.items():
    assert getattr(saved_meta, attribute) == expected, attribute

# Save an old-to-new name key with the complete definitions.
variable_names = pd.DataFrame({
    "ShortName": spss_data.columns,
    "OriginalName": source_data.columns,
    "Definition": [
        labels.get(name) for name in spss_data.columns
    ],
})
variable_names.to_csv(
    name_key_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved SPSS dataset: {output_path}")
print(f"Saved variable name key: {name_key_path}")
print(f"Rows: {len(spss_data)}; variables: {spss_data.shape[1]}")
print(
    "Maximum variable name length:",
    max(map(len, spss_data.columns)),
)

display(variable_names)
display(
    spss_data[
        ["RecID", "CondLbl", "DispQ2", "NAOIFix", "Cal"]
    ].head(10)
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat
from IPython.display import display

study_dir = OUTPUT_DIR

source_path = study_dir / "Cookie_eye_tracking_with_survey_8char.sav"
output_path = study_dir / "Cookie_eye_tracking_with_survey_visits.sav"

# Read the dataset and preserve its SPSS metadata.
spss_data, meta = pyreadstat.read_sav(
    str(source_path),
    apply_value_formats=False,
    user_missing=True,
)

# These existing duration columns are already in seconds.
duration_columns = [f"A{i}VisS" for i in range(5)]
count_columns = [f"A{i}VisN" for i in range(5)]

required_columns = duration_columns + count_columns + ["AOIVisN"]
missing_columns = set(required_columns) - set(spss_data.columns)

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {sorted(missing_columns)}"
    )

durations = spss_data[duration_columns].apply(
    pd.to_numeric, errors="raise"
)
counts = spss_data[count_columns].apply(
    pd.to_numeric, errors="raise"
)

# Validate the source measures before calculating totals.
if (
    not np.isfinite(durations.to_numpy()).all()
    or not np.isfinite(counts.to_numpy()).all()
    or durations.lt(0).any().any()
    or counts.lt(0).any().any()
    or counts.mod(1).ne(0).any().any()
):
    raise ValueError(
        "Visit durations and counts must be nonmissing and "
        "nonnegative; counts must be integers."
    )

if not counts.sum(axis=1).eq(spss_data["AOIVisN"]).all():
    raise ValueError(
        "AOIVisN does not equal the sum of the five AOI visit counts."
    )

# Total visit duration across AOIs 0-4, in seconds.
spss_data["AOIVisS"] = durations.sum(axis=1, min_count=5)

# Mean duration per visit: every visit receives equal weight.
# The mean is missing when there are no visits.
spss_data["AOIVisAv"] = (
    spss_data["AOIVisS"]
    / spss_data["AOIVisN"].where(spss_data["AOIVisN"].gt(0))
)

# Preserve existing variable definitions and add the new definitions.
labels = dict(meta.column_names_to_labels)
labels.update({
    "AOIVisS": (
        "Sum of native total visit durations for AOI positions 0-4, "
        "in seconds."
    ),
    "AOIVisAv": (
        "AOIVisS divided by AOIVisN: mean duration per AOI visit "
        "across positions 0-4, in seconds. "
        "Missing when there are no visits."
    ),
})

measures = dict(meta.variable_measure)
measures.update({
    "AOIVisS": "scale",
    "AOIVisAv": "scale",
})

formats = dict(meta.original_variable_types)
formats.update({
    "AOIVisS": "F16.3",
    "AOIVisAv": "F16.6",
})

# Export the complete dataset with its SPSS metadata.
pyreadstat.write_sav(
    spss_data,
    str(output_path),
    file_label=meta.file_label or "Cookie eye-tracking and survey data",
    column_labels=labels,
    variable_value_labels=meta.variable_value_labels,
    missing_ranges=meta.missing_ranges,
    variable_measure=measures,
    variable_format=formats,
    variable_display_width=meta.variable_display_width,
    note=meta.notes or None,
)

# Read the export back to verify values and definitions.
saved_data, saved_meta = pyreadstat.read_sav(
    str(output_path),
    apply_value_formats=False,
    user_missing=True,
)

pd.testing.assert_frame_equal(
    spss_data.reset_index(drop=True),
    saved_data,
    check_dtype=False,
    check_exact=True,
)

assert saved_meta.column_names_to_labels == labels

print(f"Saved: {output_path}")
print(
    f"Participants: {len(saved_data)}; "
    f"variables: {saved_data.shape[1]}"
)

display(
    saved_data[
        ["RecID", "CondLbl", "AOIVisN", "AOIVisS", "AOIVisAv"]
    ].head(10)
)